In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import re

In [ ]:
fdata = {
    '6x6' : None,
    '8x8' : None,
    '10x10' : None
}

In [ ]:
for k, v in fdata.items():
    fdata[k] = np.load(f'output_{k}.npz', allow_pickle=True)

In [ ]:
ftpcbatch = []
for fk, fv in fdata.items():
    for k, v in fv.items():
        mstring = re.fullmatch(r'current_tpc(\d+)_batch(\d+)', k)
        if mstring:
            # print(k, mstring[0], mstring[1], mstring[2])
            ftpcbatch.append( (int(mstring[1]), int(mstring[2])) )
ftpcbatch = set(ftpcbatch)
print(ftpcbatch)

In [ ]:
fdata['10x10']['current_tpc22_batch8'].shape

In [ ]:
def form_delta(f1, f2, k):
    if k in f1.keys():
        locs1 = f1[f'{k}_location']
        vals1 = np.max(np.abs(np.squeeze(f1[k])), axis=-1)
    else:
        print(f'{k} is missing in {f1}')
        locs1 = np.zeros((0,3), dtype=int)
        vals1 = np.zeros((0,), dtype=float)
    if k in f2.keys():  
        locs2 = f2[f'{k}_location']
        vals2 = np.max(np.abs(np.squeeze(f2[k])), axis=-1)
    else:
        print(f'{k} is missing in {f2}')
        locs2 = np.zeros((0,3), dtype=int)
        vals2 = np.zeros((0,), dtype=float)
    dk, dv, qv = diff_from_arrays(locs1, vals1, locs2, vals2)
    return dk, dv, qv

In [ ]:
from collections import defaultdict

def diff_from_arrays(keysA, valsA, keysB, valsB):
    # diff[k] = sumA(k) - sumB(k)
    diff = defaultdict(float)
    qref = defaultdict(float)

    # Add values from A
    for k, v in zip(keysA, valsA):
        kt = tuple(map(int, k))
        diff[kt] += v
        qref[kt] = v

    # Subtract values from B
    for k, v in zip(keysB, valsB):
        kt = tuple(map(int, k))
        diff[kt] -= v
        if kt not in qref.keys():
            qref[kt] = -1

    # Convert back to two arrays
    diff_keys = list(diff.keys())
    diff_vals = [diff[k] for k in diff_keys]
    qvals = [qref[k] for k in diff_keys]
    return diff_keys, diff_vals, qvals

In [ ]:
locs = defaultdict(list)
diff = defaultdict(list)
qref = defaultdict(list)
diff_cat = defaultdict(np.ndarray)
qref_cat = defaultdict(np.ndarray)
for fk in ['8x8', '6x6']:
    for itpc, ibatch in ftpcbatch:
        k = f'current_tpc{itpc}_batch{ibatch}'
        kd, kv, qv = form_delta(fdata[fk], fdata['10x10'], k)
    # print(kv[:10], k)
        diff[fk].append(kv)
        qref[fk].append(qv)
        locs[fk].append(kd)
    diff_cat[fk] = np.hstack(diff[fk])
    qref_cat[fk] = np.hstack(qref[fk])

In [ ]:
plt.hist(np.concat(diff['8x8']), bins=100, label='8x8', alpha=0.5)
plt.hist(np.concat(diff['6x6']), bins=100, label='6x6', alpha=0.5)
plt.legend()
plt.yscale('log')

In [ ]:
dqcut = 0.5
for fk in ['8x8', '6x6']:
    mask = diff_cat[fk] > dqcut
    plt.hist(qref_cat[fk][mask], bins=100, label=fk, alpha=0.5)
plt.title(f'dq > {dqcut}')
plt.legend()
plt.yscale('log')

In [ ]:
dqcut = 0.1
for fk in ['8x8', '6x6']:
    mask = diff_cat[fk] > dqcut
    plt.hist(qref_cat[fk][mask], bins=100, label=fk, alpha=0.5)
plt.title(f'dq > {dqcut}')
plt.legend()
plt.yscale('log')

In [ ]:
dqmax = 0.1
for fk in ['8x8', '6x6']:
    mask = diff_cat[fk] < dqmax
    plt.hist(qref_cat[fk][mask], bins=100, label=fk, alpha=0.5)
plt.title(f'dq < {dqmax}')
plt.legend()
plt.yscale('log')

In [ ]:
np.mean(diff_cat['8x8']), np.std(diff_cat['8x8']), np.std(diff_cat['6x6'])

In [ ]:
np.diff(np.sort(np.unique(np.array(locs['8x8'][0])[:,-1])))